IMPORTS

In [9]:
import pandas as pd 
import numpy as np 

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import joblib

LOAD DATA

In [6]:
df = pd.read_csv("../data/processed/churn_processed.csv")

print("Shape:", df.shape)
df.head()

Shape: (7043, 24)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,num_services,contract_risk,engagement_score
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,New,1,2,1.381833
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,One year,No,Mailed check,56.95,1889.50,No,Mid,3,1,6.402833
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,New,3,2,3.705167
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,Loyal,3,1,7.173000
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,New,1,2,1.873667


LOAD SAVED MODEL

In [7]:
model = joblib.load("../models/xgb_churn_model.pkl")
print("Model loaded")

Model loaded


PREPARE DATA FOR PREDICTION

In [10]:
df_model = df.copy()

#Encode target (just for consistency, not used here)
df_model['Churn'] = df_model['Churn'].map({'Yes':1 , 'No':0})

#Encode categorical features
cat_cols = df_model.select_dtypes(include=['object']).columns

le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col]) 

C:\Users\mehta\AppData\Local\Temp\ipykernel_3104\2625934569.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_model.select_dtypes(include=['object']).columns


GET CHURN PROBABILITY

In [11]:
X = df_model.drop("Churn", axis=1)

df["Churn_probability"] = model.predict_proba(X)[:,1]

df[["Churn_probability"]].head()

,Churn_probability
0,0.830421
1,0.041556
2,0.640282
3,0.032011
4,0.831514


CHECK DISTRIBUTION

In [13]:
df['Churn_probability'].describe()

count    7043.000000
mean        0.383516
std         0.324325
min         0.001109
25%         0.057098
50%         0.317831
75%         0.692542
max         0.979738
Name: Churn_probability, dtype: float64

FILTER HIGH-RISK CUSTOMERS

In [14]:
high_risk = df[df['Churn_probability'] > 0.7].copy()

print("High-risk customers:", high_risk.shape)

High-risk customers: (1726, 25)


FEATURES FOR CLUSTERING

In [15]:
cluster_features = high_risk[[
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'num_services',
    'engagement_score'
]]

SCALE DATA

In [16]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(cluster_features)

APPLY K-MEANS

In [17]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

high_risk['cluster'] = kmeans.fit_predict(scaled_data)

high_risk.value_counts()

gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  MultipleLines     InternetService  OnlineSecurity       OnlineBackup         DeviceProtection     TechSupport          StreamingTV          StreamingMovies      Contract        PaperlessBilling  PaymentMethod     MonthlyCharges  TotalCharges  Churn  tenure_group  num_services  contract_risk  engagement_score  Churn_probability  cluster
Male    0              No       No          1       Yes           No                No               No internet service  No internet service  No internet service  No internet service  No internet service  No internet service  Month-to-month  No                Mailed check      20.15           20.15         Yes    New           1             2              1.284833          0.709055           0          2
                                                                                    DSL              No                   No                   No                   No                   No

CLUSTER ANALYSIS

In [20]:
cluster_summary = high_risk.select_dtypes(include=['number']).groupby(high_risk['cluster']).mean()

cluster_summary

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,num_services,contract_risk,engagement_score,Churn_probability
cluster,,,,,,,,
0,0.242503,4.091265,59.428357,243.752934,1.522816,1.997392,2.458038,0.839217
1,0.320937,13.122590,89.201584,1141.996350,3.790634,1.997245,5.776199,0.852359
2,0.429185,47.128755,101.272747,4792.477468,5.476395,1.690987,10.416519,0.777168


SAVE SEGMENTED DATA

In [21]:
high_risk.to_csv("../data/processed/high_risk_customers.csv", index=False)

print(" Segmented data saved!")

 Segmented data saved!


SAVE KMEANS MODEL AND SCALER 

In [24]:
import joblib

# Save KMeans model
joblib.dump(kmeans, "../models/kmeans_segmentation.pkl")

# Save scaler
joblib.dump(scaler, "../models/scaler.pkl")

print("KMeans and scaler saved successfully!")

KMeans and scaler saved successfully!
